# 08_pipeline.ipynb

## Install SageMaker SDK v2

This notebook uses the classic SageMaker Python SDK v2 because the pipeline examples rely on modules such as `sagemaker.processing`, `sagemaker.workflow.steps`, and `sagemaker.sklearn.processing`.

If you run the installation cell below, restart the notebook kernel before continuing.

In [ ]:
%pip install --upgrade "sagemaker>=2,<3"

## Import Project Modules

The project source code is organized in the `src/` directory at the repository root.

Because this notebook is executed from the `notebooks/` directory, the project root is added to the Python path so that modules from the `src` package can be imported.

In [ ]:
import sys

sys.path.append("..")

## Import Dependencies

This section imports the required Python, AWS, and SageMaker libraries. These dependencies are used to define and run the SageMaker Pipeline.

In [ ]:
import json

import boto3
import sagemaker

from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.functions import Join
from sagemaker.workflow.parameters import ParameterInteger, ParameterString
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.workflow.steps import TrainingStep

from src.config import (
    AWS_REGION,
    BUCKET_NAME,
    RAW_DATA_KEY,
    RAW_DATA_S3_URI,
    PIPELINE_NAME,
    PIPELINE_PREFIX,
    PIPELINE_PROCESSED_DATA_S3_URI,
    PIPELINE_PROCESSED_PREFIX,
    PIPELINE_TRAIN_PREFIX,
    PIPELINE_VALIDATION_PREFIX,
    PIPELINE_TEST_PREFIX,
    PREPROCESSING_SCRIPT_PATH,
    TRAINING_SCRIPT_PATH,
    EVALUATION_SCRIPT_PATH,
    PROCESSING_INPUT_PATH,
    PROCESSING_TRAIN_PATH,
    PROCESSING_VALIDATION_PATH,
    PROCESSING_TEST_PATH,
)

## Initialize AWS Clients

This section initializes the AWS and SageMaker clients used throughout the notebook. It also creates the SageMaker Pipeline session and retrieves the execution role.

In [ ]:
boto_session = boto3.Session(region_name=AWS_REGION)

s3 = boto_session.client("s3")
sm = boto_session.client("sagemaker")

sagemaker_session = sagemaker.session.Session(
    boto_session=boto_session
)

pipeline_session = PipelineSession(
    boto_session=boto_session
)

role = sagemaker.get_execution_role()

print(f"Region: {AWS_REGION}")
print(f"Bucket: {BUCKET_NAME}")
print(f"Role: {role}")

## Define Pipeline Parameters

This section defines reusable pipeline parameters. Parameters make the pipeline more flexible because values such as the input data path or instance type can be changed when starting a new execution.

In [ ]:
input_data = ParameterString(
    name="InputData",
    default_value=RAW_DATA_S3_URI,
)

processing_instance_type = ParameterString(
    name="ProcessingInstanceType",
    default_value="ml.t3.medium",
)

processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount",
    default_value=1,
)

processed_data_output_path = ParameterString(
    name="ProcessedDataOutputPath",
    default_value=PIPELINE_PROCESSED_DATA_S3_URI,
)

training_instance_type = ParameterString(
    name="TrainingInstanceType",
    default_value="ml.m5.large",
)

training_instance_count = ParameterInteger(
    name="TrainingInstanceCount",
    default_value=1,
)

model_output_path = ParameterString(
    name="ModelOutputPath",
    default_value=f"s3://{BUCKET_NAME}/{PIPELINE_PREFIX}/model",
)

evaluation_output_path = ParameterString(
    name="EvaluationOutputPath",
    default_value=f"s3://{BUCKET_NAME}/{PIPELINE_PREFIX}/evaluation",
)

## Define Data Processing Step

This section defines the data processing part of the pipeline. It uses a SageMaker Processing job to run the preprocessing script, create train, validation, and test datasets, and store the outputs in Amazon S3.

### Define Processor

This section defines the SageMaker Processing environment. The processor specifies which framework image, instance type, and execution role should be used to run the preprocessing script.

In [ ]:
sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type=processing_instance_type,
    instance_count=processing_instance_count,
    base_job_name="credit-risk-preprocess",
    sagemaker_session=pipeline_session,
)

### Define Processing Inputs and Outputs

This subsection defines where the Processing job reads its input data from and where it writes its output data. The raw dataset is loaded from Amazon S3, while the processed datasets are written locally and uploaded back to Amazon S3 by SageMaker.

In [ ]:
processing_inputs = [
    ProcessingInput(
        input_name="raw-data",
        source=input_data,
        destination=PROCESSING_INPUT_PATH,
    )
]

In [ ]:
processing_outputs = [
    ProcessingOutput(
        output_name="train",
        source=PROCESSING_TRAIN_PATH,
        destination=Join(
            on="/",
            values=[
                processed_data_output_path,
                "train",
            ],
        ),
    ),
    ProcessingOutput(
        output_name="validation",
        source=PROCESSING_VALIDATION_PATH,
        destination=Join(
            on="/",
            values=[
                processed_data_output_path,
                "validation",
            ],
        ),
    ),
    ProcessingOutput(
        output_name="test",
        source=PROCESSING_TEST_PATH,
        destination=Join(
            on="/",
            values=[
                processed_data_output_path,
                "test",
            ],
        ),
    ),
]

### Define ProcessingStep

This subsection combines the processor, inputs, outputs, script, and script arguments into a SageMaker Pipeline step. The step runs the preprocessing script as part of the pipeline workflow.

In [ ]:
step_args = sklearn_processor.run(
    inputs=processing_inputs,
    outputs=processing_outputs,
    code="../src/preprocess_simplified.py",
    arguments=[
        "--input-data", "/opt/ml/processing/input",
        "--train-output", "/opt/ml/processing/train/train.csv",
        "--validation-output", "/opt/ml/processing/validation/validation.csv",
        "--test-output", "/opt/ml/processing/test/test.csv",
        "--test-size", "0.2",
        "--validation-size", "0.2",
        "--random-state", "42",
    ],
)

step_process = ProcessingStep(
    name="PreprocessCreditRiskData",
    step_args=step_args,
)

## Define Model Training Step

This section defines the model training part of the pipeline. It configures the training environment, provides the processed datasets as input, and executes the reusable training script to produce a trained model artifact.

### Define Estimator

This subsection defines the SageMaker training environment. The estimator specifies the training container, execution role, instance type, and output location for the trained model.

In [ ]:
sklearn_estimator = SKLearn(
    entry_point="train.py",
    source_dir="../src",
    framework_version="1.2-1",
    role=role,
    instance_type=training_instance_type,
    instance_count=1,
    output_path=model_output_path,
    sagemaker_session=pipeline_session,
)

### Define Training Inputs

This subsection connects the outputs of the data processing step to the training step. The training script receives the training and validation datasets generated by the preprocessing step.

In [ ]:
training_inputs = {
    "train": TrainingInput(
        s3_data=step_process.properties.ProcessingOutputConfig.Outputs[
            "train"
        ].S3Output.S3Uri
    ),
    "validation": TrainingInput(
        s3_data=step_process.properties.ProcessingOutputConfig.Outputs[
            "validation"
        ].S3Output.S3Uri
    ),
}

### Define TrainingStep

This subsection creates the SageMaker Training step and executes the reusable training script.

In [ ]:
step_train = TrainingStep(
    name="TrainCreditRiskModel",
    estimator=sklearn_estimator,
    inputs=training_inputs,
)

## Define Model Evaluation Step

This section evaluates the trained model using the test dataset created during preprocessing. The evaluation step generates a report containing the model performance metrics.

### Define Processor

This subsection defines the processing environment used to execute the evaluation script.

In [ ]:
evaluation_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="credit-risk-evaluation",
    sagemaker_session=pipeline_session,
)

### Define Evaluation Inputs and Outputs

This subsection provides the trained model and the test dataset to the evaluation script. The generated evaluation report is stored in Amazon S3.

In [ ]:
evaluation_inputs = [
    ProcessingInput(
        source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
        destination="/opt/ml/processing/model",
    ),
    ProcessingInput(
        source=step_process.properties.ProcessingOutputConfig.Outputs[
            "test"
        ].S3Output.S3Uri,
        destination="/opt/ml/processing/test",
    ),
]

evaluation_outputs = [
    ProcessingOutput(
        output_name="evaluation",
        source="/opt/ml/processing/evaluation",
        destination=evaluation_output_path,
    ),
]

### Define EvaluationStep

This subsection creates the evaluation step and executes the reusable evaluation script.

In [ ]:
step_args = evaluation_processor.run(
    inputs=evaluation_inputs,
    outputs=evaluation_outputs,
    code="../src/evaluate.py",
    arguments=[
        "--model",
        "/opt/ml/processing/model/model.pkl",
        "--test-data",
        "/opt/ml/processing/test/test.csv",
        "--evaluation-output",
        "/opt/ml/processing/evaluation/evaluation.json",
    ],
)

step_evaluate = ProcessingStep(
    name="EvaluateCreditRiskModel",
    step_args=step_args,
)

## Create SageMaker Pipeline

This section creates the pipeline object and combines the defined parameters and steps. At this point, the pipeline exists as a reusable workflow definition.

Afterwards, the pipeline is created or updated if it already exists. This makes the pipeline available for execution in the AWS account.

In [ ]:
pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=[
        input_data,
        processing_instance_type,
        processing_instance_count,
        processed_data_output_path,
        training_instance_type,
        training_instance_count,
        model_output_path,
        evaluation_output_path,
    ],
    steps=[
        step_process,
        step_train,
        step_evaluate,
    ],
    sagemaker_session=pipeline_session,
)

pipeline.upsert(
    role_arn=role
)

print(f"Pipeline created or updated: {PIPELINE_NAME}")

## Start Pipeline Execution

This section starts a new execution of the SageMaker Pipeline. Each execution runs the configured steps using the current pipeline parameters.

In [ ]:
execution = pipeline.start()

print(f"Started pipeline execution: {execution.arn}")

In [ ]:
execution.wait()

print("Pipeline execution completed.")

## Review Pipeline Outputs

This section reviews the outputs produced by the pipeline execution. Depending on the pipeline version, these outputs may include processed datasets, trained model artifacts, evaluation reports, or registered model metadata.

In [ ]:
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=f"{PIPELINE_PREFIX}/processed",
)

for item in response.get("Contents", []):
    print(item["Key"])

## Result

This section summarizes the outcome of the pipeline execution. The pipeline has automated the data processing workflow and stored the processed datasets in Amazon S3 for later training steps.